# Amazon Recommendation System - Data Preparation

This notebook performs:

1. Data Loading (Reviews + Metadata)
2. Data Cleaning
3. Data Merging
4. Final Dataset Preparation

This cleaned dataset will be used for:
- EDA
- Text Preprocessing
- Recommendation Models

In [1]:
# Import core libraries
import pandas as pd
import numpy as np

In [2]:
# Define paths
DATA_RAW = "../data/raw/"
DATA_PROCESSED = "../data/processed/"

# Load data
reviews = pd.read_csv(DATA_RAW + "Appliances.csv", low_memory=False)
meta = pd.read_json(DATA_RAW + "meta_Appliances.json", lines=True)

# Merge
df = reviews.merge(meta, on="asin", how="left")

# Save merged (before cleaning)
df.to_csv(DATA_PROCESSED + "merged_data.csv", index=False)

In [3]:
# Load Merged Data
df.head()

,overall,vote,verified,reviewTime,reviewerID,asin,style,reviewerName,reviewText,summary,...,feature,rank,also_view,details,main_cat,similar_item,date,price,imageURL,imageURLHighRes
0,5,2,False,"11 27, 2013",A3NHUQ33CFH3VM,1118461304,{'Format:': ' Hardcover'},Greeny,Not one thing in this book seemed an obvious o...,Clear on what leads to innovation,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,5,NaN,False,"11 1, 2013",A3SK6VNBQDNBJE,1118461304,{'Format:': ' Kindle Edition'},Leif C. Ulstrup,I have enjoyed Dr. Alan Gregerman's weekly blo...,Becoming more innovative by opening yourself t...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5,NaN,False,"10 10, 2013",A3SOFHUR27FO3K,1118461304,{'Format:': ' Hardcover'},Harry Gilbert Miller III,Alan Gregerman believes that innovation comes ...,The World from Different Perspectives,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,5,NaN,False,"10 9, 2013",A1HOG1PYCAE157,1118461304,{'Format:': ' Hardcover'},Rebecca Ripley,"Alan Gregerman is a smart, funny, entertaining...",Strangers are Your New Best Friends,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,10,False,"09 7, 2013",A26JGAM6GZMM4V,1118461304,{'Format:': ' Hardcover'},Robert Morris,"As I began to read this book, I was again remi...","How and why it is imperative to engage, learn ...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Initial Inspection

In [4]:
df.shape
df.info()
df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 616894 entries, 0 to 616893
Data columns (total 30 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   overall          616894 non-null  int64 
 1   vote             66694 non-null   str   
 2   verified         616894 non-null  bool  
 3   reviewTime       616894 non-null  str   
 4   reviewerID       616894 non-null  str   
 5   asin             616894 non-null  str   
 6   style            145006 non-null  str   
 7   reviewerName     616810 non-null  str   
 8   reviewText       616536 non-null  str   
 9   summary          616752 non-null  str   
 10  unixReviewTime   616894 non-null  int64 
 11  image            9413 non-null    str   
 12  category         615747 non-null  object
 13  tech1            615747 non-null  str   
 14  description      615747 non-null  object
 15  fit              615747 non-null  str   
 16  title            615747 non-null  str   
 17  also_buy         6157

overall                 0
vote               550200
verified                0
reviewTime              0
reviewerID              0
asin                    0
style              471888
reviewerName           84
reviewText            358
summary               142
unixReviewTime          0
image              607481
category             1147
tech1                1147
description          1147
fit                  1147
title                1147
also_buy             1147
tech2                1147
brand                1147
feature              1147
rank                 1147
also_view            1147
details              1147
main_cat             1147
similar_item         1147
date                 1147
price                1147
imageURL             1147
imageURLHighRes      1147
dtype: int64

### Fix List-Type Columns

Convert list-type fields (category, description) into usable formats.

In [5]:
import ast

# Convert string → list safely
df["category"] = df["category"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else []
)

df["description"] = df["description"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else []
)

# Now create columns properly
df["category_list"] = df["category"]

df["category_text"] = df["category_list"].apply(
    lambda x: " ".join(x) if isinstance(x, list) and len(x) > 0 else "unknown"
)

df["description"] = df["description"].apply(
    lambda x: " ".join(x) if isinstance(x, list) and len(x) > 0 else "unknown"
)

### Handle Missing Values

In [7]:
# Drop critical missing fields
df = df.dropna(subset=["reviewerID", "asin", "reviewText"])

# Fill optional fields
df["summary"] = df["summary"].fillna("")
df["title"] = df["title"].fillna("")
df["brand"] = df["brand"].fillna("")

### Remove Duplicate Entries

In [8]:
df = df.drop_duplicates(subset=["reviewerID", "asin", "reviewText"])

### Remove Empty Reviews

In [9]:
df = df[df["reviewText"].str.strip() != ""]

### Convert Datatypes

In [10]:
# Convert timestamp
df["reviewTime"] = pd.to_datetime(df["unixReviewTime"], unit="s", errors="coerce")

# Ensure rating is numeric
df["overall"] = df["overall"].astype(float)

### Final Column Selection

Keep only relevant columns for modeling.

In [11]:
df = df[[
    "reviewerID",
    "asin",
    "overall",
    "reviewText",
    "summary",
    "reviewTime",
    "title",
    "brand",
    "category_list",
    "category_text",
    "description"
]]

df.head()

,reviewerID,asin,overall,reviewText,summary,reviewTime,title,brand,category_list,category_text,description
0,A3NHUQ33CFH3VM,1118461304,5.0,Not one thing in this book seemed an obvious o...,Clear on what leads to innovation,2013-11-27,,,[],unknown,unknown
1,A3SK6VNBQDNBJE,1118461304,5.0,I have enjoyed Dr. Alan Gregerman's weekly blo...,Becoming more innovative by opening yourself t...,2013-11-01,,,[],unknown,unknown
2,A3SOFHUR27FO3K,1118461304,5.0,Alan Gregerman believes that innovation comes ...,The World from Different Perspectives,2013-10-10,,,[],unknown,unknown
3,A1HOG1PYCAE157,1118461304,5.0,"Alan Gregerman is a smart, funny, entertaining...",Strangers are Your New Best Friends,2013-10-09,,,[],unknown,unknown
4,A26JGAM6GZMM4V,1118461304,5.0,"As I began to read this book, I was again remi...","How and why it is imperative to engage, learn ...",2013-09-07,,,[],unknown,unknown


### Final Dataset Check

In [12]:
df.shape
df.isnull().sum()
df.info()

<class 'pandas.DataFrame'>
Index: 590985 entries, 0 to 616893
Data columns (total 11 columns):
 #   Column         Non-Null Count   Dtype        
---  ------         --------------   -----        
 0   reviewerID     590985 non-null  str          
 1   asin           590985 non-null  str          
 2   overall        590985 non-null  float64      
 3   reviewText     590985 non-null  str          
 4   summary        590985 non-null  str          
 5   reviewTime     590985 non-null  datetime64[s]
 6   title          590985 non-null  str          
 7   brand          590985 non-null  str          
 8   category_list  590985 non-null  object       
 9   category_text  590985 non-null  str          
 10  description    590985 non-null  str          
dtypes: datetime64[s](1), float64(1), object(1), str(8)
memory usage: 54.1+ MB


### Save Clean Dataset

In [13]:
df.to_csv(DATA_PROCESSED + "cleaned_data.csv", index=False)

### Summary

- Cleaned missing values
- Removed duplicates
- Processed category and description fields
- Converted timestamps
- Prepared dataset for EDA and modeling